# 개발 환경

In [107]:
import sys
import torch

print(f"Python       : {sys.version}")
print(f"PyTorch      : {torch.__version__}")
print(f"PyTorch CUDA : {torch.version.cuda}")
print(f"CUDA usable  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")

Python       : 3.12.13 (main, Apr  7 2026, 23:13:34) [GCC 13.3.0]
PyTorch      : 2.14.0+cu132
PyTorch CUDA : 13.2
CUDA usable  : True
GPU           : NVIDIA GeForce RTX 5070 Ti


# 변수 초기화

In [113]:
import torch

from markov_reward_process import MarkovRewardProcess

In [114]:
states = [
        "Class1",
        "Class2",
        "Class3",
        "Pass",
        "Pub",
        "Facebook",
        "Sleep"
    ]

    
state_transition_matrix = torch.tensor(
    [
        [0.0, 0.5, 0.0, 0.0, 0.0, 0.5, 0.0],
        [0.0, 0.0, 0.8, 0.0, 0.0, 0.0, 0.2],
        [0.0, 0.0, 0.0, 0.6, 0.4, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
        [0.2, 0.4, 0.4, 0.0, 0.0, 0.0, 0.0],
        [0.1, 0.0, 0.0, 0.0, 0.0, 0.9, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
    ]
)

rewards = torch.tensor(
    [
        -2,
        -2,
        -2,
        10,
        1,
        -1,
        0
    ]
)

discount_factor = 0.9

In [115]:
print(states)
print(state_transition_matrix)
print(rewards)

['Class1', 'Class2', 'Class3', 'Pass', 'Pub', 'Facebook', 'Sleep']
tensor([[0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.5000, 0.0000],
        [0.0000, 0.0000, 0.8000, 0.0000, 0.0000, 0.0000, 0.2000],
        [0.0000, 0.0000, 0.0000, 0.6000, 0.4000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000],
        [0.2000, 0.4000, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000]])
tensor([-2, -2, -2, 10,  1, -1,  0])


In [116]:
print(state_transition_matrix.sum(dim=1))

tensor([1., 1., 1., 1., 1., 1., 1.])


In [117]:
state_to_index = {
    
    #dictionary 생성
    state: index
    
    # enumerate(): 튜플 생성
    for index, state in enumerate(states)
}

print(state_to_index)

{'Class1': 0, 'Class2': 1, 'Class3': 2, 'Pass': 3, 'Pub': 4, 'Facebook': 5, 'Sleep': 6}


# mrp.sample_next_state

In [102]:
state = "Class1"
state_index = state_to_index[state]

P = state_transition_matrix

print(states)
print(state_to_index,"\n")

print(state_index,"\n")

print(P[state_index])

['Class1', 'Class2', 'Class3', 'Pass', 'Pub', 'Facebook', 'Sleep']
{'Class1': 0, 'Class2': 1, 'Class3': 2, 'Pass': 3, 'Pub': 4, 'Facebook': 5, 'Sleep': 6} 

0 

tensor([0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.5000, 0.0000])


In [103]:
generator = torch.Generator()
generator.manual_seed(42)

## 매우 중요

In [ ]:
current_state = "Pub"
current_index = state_to_index[current_state]

probabilities = P[current_index]

# 매우 중요!!!
next_index = torch.multinomial(
    probabilities,
    num_samples=1,
    generator=generator,
).item()

next_state = states[next_index]


print(state_to_index,"\n")

print(current_state)
print(current_index,"\n")

print(probabilities,"\n")

print(next_state)
print(next_index)

{'Class1': 0, 'Class2': 1, 'Class3': 2, 'Pass': 3, 'Pub': 4, 'Facebook': 5, 'Sleep': 6} 

Pub
4 

tensor([0.2000, 0.4000, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000]) 

Class2
1


In [105]:
def sample_next_state(state):
    state_index = state_to_index[state]

    probabilities = P[state_index]

    next_index = torch.multinomial(
        probabilities,
        num_samples=1,
        generator=generator,
    ).item()

    return states[next_index]

In [106]:
print(sample_next_state("Pub"))
print(sample_next_state("Pub"))
print(sample_next_state("Pub"))
print(sample_next_state("Pub"))
print(sample_next_state("Pub"))
print(sample_next_state("Pub"))

Class2
Class3
Class3
Class2
Class3
Class2


# mrp.get_reward()

In [65]:
def get_reward(state):
    state_index = state_to_index[state]

    return rewards[state_index]

In [61]:
print(get_reward("Pass"))
print(get_reward("Pub"))
print(get_reward("Facebook"), "\n")

print(get_reward("Pass").item())

tensor(10)
tensor(1)
tensor(-1) 

10


# mrp.simulate()

In [73]:
current_state = "Pub"

reward = get_reward(current_state)

next_state = sample_next_state(current_state)

print("Current:", current_state)
print("Reward:", reward.item())
print("Next:", next_state)

Current: Pub
Reward: 1
Next: Class3


In [83]:
current_state = "Class1"

for step in range(10):
    reward = get_reward(current_state)
    next_state = sample_next_state(current_state)

    print(
        step,
        current_state,
        reward.item(),
        next_state,
    )

    current_state = next_state

0 Class1 -2 Class2
1 Class2 -2 Class3
2 Class3 -2 Pass
3 Pass 10 Sleep
4 Sleep 0 Sleep
5 Sleep 0 Sleep
6 Sleep 0 Sleep
7 Sleep 0 Sleep
8 Sleep 0 Sleep
9 Sleep 0 Sleep


In [87]:
trajectory = []

current_state = "Class1"

for step in range(10):
    reward = get_reward(current_state)
    next_state = sample_next_state(current_state)

    trajectory.append(
        {
            "step": step,
            "state": current_state,
            "reward": reward.item(),
            "next_state": next_state,
        }
    )

    current_state = next_state

print(trajectory)

[{'step': 0, 'state': 'Class1', 'reward': -2, 'next_state': 'Facebook'}, {'step': 1, 'state': 'Facebook', 'reward': -1, 'next_state': 'Class1'}, {'step': 2, 'state': 'Class1', 'reward': -2, 'next_state': 'Class2'}, {'step': 3, 'state': 'Class2', 'reward': -2, 'next_state': 'Sleep'}, {'step': 4, 'state': 'Sleep', 'reward': 0, 'next_state': 'Sleep'}, {'step': 5, 'state': 'Sleep', 'reward': 0, 'next_state': 'Sleep'}, {'step': 6, 'state': 'Sleep', 'reward': 0, 'next_state': 'Sleep'}, {'step': 7, 'state': 'Sleep', 'reward': 0, 'next_state': 'Sleep'}, {'step': 8, 'state': 'Sleep', 'reward': 0, 'next_state': 'Sleep'}, {'step': 9, 'state': 'Sleep', 'reward': 0, 'next_state': 'Sleep'}]


# mrp.discounted_return()

In [95]:
gamma = discount_factor

G = (
    trajectory[0]["reward"]
    + gamma * trajectory[1]["reward"]
    + gamma**2 * trajectory[2]["reward"]
)

print(trajectory[0]["state"])
print(trajectory[0]["reward"], "\n")

print(trajectory[1]["state"])
print(trajectory[1]["reward"])
print(gamma * trajectory[1]["reward"], "\n")

print(trajectory[2]["state"])
print(trajectory[2]["reward"])
print(gamma**2 * trajectory[2]["reward"], "\n")

print(G)

Class1
-2 

Facebook
-1
-0.9 

Class1
-2
-1.62 

-4.52


In [98]:
total_return = torch.tensor(
    0.0,
)

for t, transition in enumerate(trajectory):
    reward = transition["reward"]

    total_return += (
        gamma ** t
    ) * reward


print(trajectory[0]["state"])
print(trajectory[0]["reward"], "\n")

print(trajectory[1]["state"])
print(trajectory[1]["reward"])
print(gamma * trajectory[1]["reward"], "\n")

print(trajectory[2]["state"])
print(trajectory[2]["reward"])
print(gamma**2 * trajectory[2]["reward"], "\n")

print(total_return.item())

Class1
-2 

Facebook
-1
-0.9 

Class1
-2
-1.62 

-5.977999687194824


# MarkovRewardProcess class 객체 만들기

In [121]:
import torch


class MarkovRewardProcess:

    def __init__(
        self,
        states,
        transition_matrix,
        rewards,
        gamma,
        seed=None,
    ):
        self.states = list(states)

        self.P = torch.as_tensor(
            transition_matrix,
            dtype=torch.float64,
        )

        self.R = torch.as_tensor(
            rewards,
            dtype=torch.float64,
        )

        self.gamma = gamma

        self.n_states = len(self.states)

        self.state_to_index = {
            state: index
            for index, state in enumerate(self.states)
        }

        self.generator = torch.Generator()

        if seed is not None:
            self.generator.manual_seed(seed)

In [124]:
    mrp = MarkovRewardProcess(
        states=states,
        transition_matrix=state_transition_matrix,
        rewards=rewards,
        gamma=discount_factor,
        seed=42,
    )

# Bellman Equation

In [ ]:
values = torch.zeros(
    len(mrp.states),
    dtype=torch.float64,
)

print(mrp.states, "\n")

print(values)

['Class1', 'Class2', 'Class3', 'Pass', 'Pub', 'Facebook', 'Sleep'] 

tensor([0., 0., 0., 0., 0., 0., 0.], dtype=torch.float64)


In [ ]:
new_values = (
    mrp.R
    + mrp.gamma * (mrp.P @ values)
)

print(mrp.R, "\n")

print(mrp.P, "\n")

print(values, "\n")

print(new_values)

tensor([-2., -2., -2., 10.,  1., -1.,  0.], dtype=torch.float64) 

tensor([[0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.5000, 0.0000],
        [0.0000, 0.0000, 0.8000, 0.0000, 0.0000, 0.0000, 0.2000],
        [0.0000, 0.0000, 0.0000, 0.6000, 0.4000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000],
        [0.2000, 0.4000, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000]],
       dtype=torch.float64) 

tensor([0., 0., 0., 0., 0., 0., 0.], dtype=torch.float64) 

tensor([-2., -2., -2., 10.,  1., -1.,  0.], dtype=torch.float64)


## Bellman equation 반복 학습으로 구하기

In [ ]:
values = torch.zeros(
    len(mrp.states),
    dtype=torch.float64,
)

for _ in range(10000):
    new_values = (
        mrp.R
        + mrp.gamma * (mrp.P @ values)
    )

    # 가장 큰 변화량
    error = torch.max(
        torch.abs(new_values - values)
    )

    values = new_values

    if error.item() < 1e-10:
        break



print(mrp.gamma, "\n")

print(error.item(), "\n")

print(values)

0.9 

8.950529206686042e-11 

tensor([-5.0127,  0.9427,  4.0870, 10.0000,  1.9084, -7.6376,  0.0000],
       dtype=torch.float64)


## Bellman equation 역행렬로 구하기

V=R+\gamma PV

V=R+\gamma PV

(I-\gamma P)V=R

In [141]:
I = torch.eye(
    len(mrp.states),
    dtype=torch.float64,
)

print(mrp.states, "\n")

print(I)

['Class1', 'Class2', 'Class3', 'Pass', 'Pub', 'Facebook', 'Sleep'] 

tensor([[1., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 1.]], dtype=torch.float64)


### 매우 중요

In [ ]:
# 매우 중요!!!
V_exact = torch.linalg.solve(
    I - mrp.gamma * mrp.P,
    mrp.R,
)

print(mrp.P, "\n")

print(mrp.R, "\n")

print(V_exact)

tensor([[0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.5000, 0.0000],
        [0.0000, 0.0000, 0.8000, 0.0000, 0.0000, 0.0000, 0.2000],
        [0.0000, 0.0000, 0.0000, 0.6000, 0.4000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000],
        [0.2000, 0.4000, 0.4000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1000, 0.0000, 0.0000, 0.0000, 0.0000, 0.9000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000]],
       dtype=torch.float64) 

tensor([-2., -2., -2., 10.,  1., -1.,  0.], dtype=torch.float64) 

tensor([-5.0127,  0.9427,  4.0870, 10.0000,  1.9084, -7.6376,  0.0000],
       dtype=torch.float64)


In [ ]:
## 두 방법 비교하기

In [143]:
print(values)
print(V_exact)

tensor([-5.0127,  0.9427,  4.0870, 10.0000,  1.9084, -7.6376,  0.0000],
       dtype=torch.float64)
tensor([-5.0127,  0.9427,  4.0870, 10.0000,  1.9084, -7.6376,  0.0000],
       dtype=torch.float64)


In [144]:
print(
    torch.abs(
        values - V_exact
    )
)

tensor([3.0480e-10, 3.2985e-11, 3.9394e-11, 0.0000e+00, 9.4102e-11, 5.4948e-10,
        0.0000e+00], dtype=torch.float64)


# if __name__ == "__main__": main()

In [ ]:
# python main.py를 실행할 경우 __name__ == "__main__" 이 된다.
# 따라서 main() 함수가 실행된다.
# 그러나, 다른 파일에서 import main 코드를 넣으면
# 그 파일이 실행될 때 __name__ == "main" 이 된다.

# 다른 파일에서 import main 코드를 넣으면
# 1. main.py 파일을 찾음
# 2. main.py의 코드를 위에서 아래로 실행함
# 3. 함수와 클래스 정의를 생성함
# 4. 최상위에 있는 일반 명령문도 실행함
# 5. 만들어진 모듈 객체를 import한 곳에 연결함

# python a.py를 실행해도 __name__ == "__main__" 이 된다.
# __name__ == "__main__" 은 프로그램의 시작점으로 실행된
# 모듈에 특별히 부여되는 것이기 때문이다.
if __name__ == "__main__":
    main()